# Canadian Pipeline Network — Interactive Map

This notebook converts a cleaned pipeline GeoDataFrame into GeoJSON and builds an interactive Mapbox GL JS map of Canadian pipeline infrastructure.

## Workflow
1. Configure dependencies, the Mapbox access token, and output settings.
2. Convert pipeline geometries and attributes to GeoJSON.
3. Download and prepare Canada's national boundary.
4. Build an outside-Canada mask.
5. Generate the interactive HTML map.

**Expected input:** a pandas DataFrame (or GeoDataFrame) named `pipelines_df` with a `geometry` column. Geometry values may be Shapely objects or WKT strings.

> Security: provide your Mapbox token through an environment variable or Colab secret. Avoid sharing notebooks containing a live access token.


## 1. Install dependencies

Run this cell in Google Colab if the packages are not already installed.

In [ ]:
%pip -q install pandas requests shapely


## 2. Imports and configuration

Set `MAPBOX_TOKEN` in the environment before running this cell. In Colab, you can use Secrets (the key name below is `MAPBOX_TOKEN`).

In [ ]:
import os
import json
import html

import requests
import pandas as pd

from shapely import wkt
from shapely.geometry import shape, mapping, box
from shapely.ops import unary_union


# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

# Prefer an environment variable or Colab Secret over hard-coding credentials.
MAPBOX_TOKEN = os.environ.get("MAPBOX_TOKEN", "")

if not MAPBOX_TOKEN:
    raise ValueError(
        "MAPBOX_TOKEN is missing. Add it as an environment variable "
        "or a Colab Secret named MAPBOX_TOKEN."
    )

OUTPUT_HTML = "pipeline_map.html"

CANADA_GEOJSON_URL = (
    "https://raw.githubusercontent.com/nvkelso/"
    "natural-earth-vector/master/geojson/"
    "ne_10m_admin_0_countries.geojson"
)

# Approximate Canadian navigation bounds: [west, south], [east, north].
CANADA_BOUNDS = [
    [-145, 49],
    [-45, 85]
]

# Pipeline attributes to include in each GeoJSON feature.
PIPELINE_PROPERTY_COLUMNS = [
    "pipeline_name",
    "company",
    "parent_company",
    "province",
    "location_description",
    "from_location",
    "to_location",
    "commodity",
    "pipeline_length_km",
    "capacity",
    "status",
    "start_date",
    "end_date",
    "segment_number",
    "licence_number",
    "project_number",
    "authority_type",
    "data_source",
    "source_dataset",
    "source_record_id"
]


## 3. Convert the pipeline DataFrame to GeoJSON

Each valid row becomes a GeoJSON `Feature`. Rows with missing, empty, or invalid geometries are skipped.

In [ ]:
def dataframe_to_geojson(df):
    """Convert a DataFrame containing Shapely or WKT geometries to GeoJSON."""
    features = []

    for _, row in df.iterrows():
        geometry = row.get("geometry")

        # Skip records without geometry.
        if geometry is None or (isinstance(geometry, float) and pd.isna(geometry)):
            continue

        # Parse WKT strings; otherwise use the supplied Shapely geometry.
        try:
            geom = wkt.loads(geometry) if isinstance(geometry, str) else geometry

            if geom is None or geom.is_empty:
                continue

        except Exception as exc:
            print(f"Skipping invalid geometry: {exc}")
            continue

        # Copy only the selected attributes that exist in the input DataFrame.
        properties = {}

        for column in PIPELINE_PROPERTY_COLUMNS:
            if column not in df.columns:
                continue

            value = row[column]

            # Convert missing values to JSON null and NumPy scalars to Python scalars.
            try:
                if pd.isna(value):
                    value = None
                elif hasattr(value, "item"):
                    value = value.item()
                elif not isinstance(value, (str, int, float, bool)):
                    value = str(value)
            except (TypeError, ValueError):
                # Keep non-scalar values representable as strings.
                value = str(value)

            properties[column] = value

        features.append({
            "type": "Feature",
            "geometry": mapping(geom),
            "properties": properties
        })

    return {
        "type": "FeatureCollection",
        "features": features
    }


## 4. Download and prepare Canada's boundary

The boundary is retrieved from the Natural Earth GeoJSON dataset and dissolved into one geometry.

In [ ]:
def get_canada_boundary():
    """Download the Natural Earth country dataset and return Canada's geometry."""
    print("Downloading Canada boundary...")

    response = requests.get(CANADA_GEOJSON_URL, timeout=60)
    response.raise_for_status()

    world_data = response.json()

    canada_features = [
        feature
        for feature in world_data["features"]
        if feature["properties"].get("ADMIN") == "Canada"
    ]

    if not canada_features:
        raise ValueError("Could not find Canada in the boundary dataset.")

    canada_geometry = unary_union([
        shape(feature["geometry"])
        for feature in canada_features
    ])

    return canada_geometry


## 5. Create the outside-Canada mask

A large world polygon is cut by Canada's geometry. The resulting polygon is used as a grey overlay, leaving Canada as a transparent hole.

In [ ]:
def create_outside_mask(canada_geometry):
    """Return a GeoJSON feature covering the world outside Canada."""
    world = box(-180, -85, 180, 85)
    outside_canada = world.difference(canada_geometry)

    return {
        "type": "Feature",
        "geometry": mapping(outside_canada),
        "properties": {
            "name": "Outside Canada"
        }
    }


## 6. Generate the interactive Mapbox map

This function assembles the GeoJSON layers and embeds them in a standalone HTML page. The generated map includes:
- A grey mask outside Canada and a Canada outline.
- Pipeline line features.
- Click popups with pipeline attributes.
- Navigation and fullscreen controls.
- An automatic initial zoom to Canada's extent.


In [ ]:
# HTML/CSS/JavaScript template used by the map generator.
# Python placeholders are replaced with serialized data in the next function.

MAP_HTML_TEMPLATE = r"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Canadian Pipeline Network</title>

    <script src="https://api.mapbox.com/mapbox-gl-js/v3.9.4/mapbox-gl.js"></script>
    <link href="https://api.mapbox.com/mapbox-gl-js/v3.9.4/mapbox-gl.css" rel="stylesheet">

    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { font-family: Arial, sans-serif; overflow: hidden; }
        #map { width: 100%; height: 100vh; }

        .map-title {
            position: absolute; top: 20px; left: 20px; z-index: 10;
            background: white; padding: 15px 20px; border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.15);
        }
        .map-title h2 { font-size: 18px; margin-bottom: 5px; }
        .map-title p { font-size: 12px; color: #666; }

        .legend {
            position: absolute; bottom: 30px; left: 20px; z-index: 10;
            background: white; padding: 15px; border-radius: 8px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.15); font-size: 13px;
        }
        .legend-item { display: flex; align-items: center; margin-top: 8px; }
        .legend-line { width: 25px; height: 4px; background: #e04444; margin-right: 10px; }
        .legend-mask { width: 25px; height: 14px; background: #999; margin-right: 10px; }

        .mapboxgl-popup-content {
            padding: 15px; border-radius: 8px; min-width: 220px; max-width: 320px;
        }
        .popup-title { font-size: 16px; font-weight: bold; margin-bottom: 10px; }
        .popup-row { font-size: 12px; margin: 6px 0; color: #444; }
        .popup-row strong { color: #111; }
    </style>
</head>

<body>
    <div class="map-title">
        <h2>Canadian Pipeline Network</h2>
        <p>Interactive pipeline infrastructure map</p>
    </div>

    <div class="legend">
        <strong>Map Legend</strong>
        <div class="legend-item"><div class="legend-line"></div>Pipeline Network</div>
        <div class="legend-item"><div class="legend-mask"></div>Outside Canada</div>
    </div>

    <div id="map"></div>

    <script>
        // Initialize Mapbox GL JS.
        mapboxgl.accessToken = "__MAPBOX_TOKEN__";

        const map = new mapboxgl.Map({
            container: "map",
            style: "mapbox://styles/mapbox/light-v11",
            center: [__CENTER_LON__, __CENTER_LAT__],
            zoom: 3,
            minZoom: 2,
            maxZoom: 15,
            maxBounds: __CANADA_BOUNDS__,
            renderWorldCopies: false
        });

        map.addControl(new mapboxgl.NavigationControl(), "top-right");
        map.addControl(new mapboxgl.FullscreenControl(), "top-right");

        // GeoJSON data prepared by the Python cells.
        const pipelineData = __PIPELINE_GEOJSON__;
        const canadaData = __CANADA_GEOJSON__;
        const outsideMask = __OUTSIDE_MASK__;

        map.on("load", function () {
            // Add Canada's boundary and the outside-country mask.
            map.addSource("canada", { type: "geojson", data: canadaData });
            map.addSource("outside-mask", { type: "geojson", data: outsideMask });

            map.addLayer({
                id: "outside-canada-fill",
                type: "fill",
                source: "outside-mask",
                paint: { "fill-color": "#999999", "fill-opacity": 0.78 }
            });

            map.addLayer({
                id: "canada-outline",
                type: "line",
                source: "canada",
                paint: {
                    "line-color": "#333333",
                    "line-width": 1.5,
                    "line-opacity": 0.9
                }
            });

            // Add pipeline features as red lines.
            map.addSource("pipelines", { type: "geojson", data: pipelineData });

            map.addLayer({
                id: "pipeline-lines",
                type: "line",
                source: "pipelines",
                layout: { "line-join": "round", "line-cap": "round" },
                paint: {
                    "line-color": "#e04444",
                    "line-width": [
                        "interpolate", ["linear"], ["zoom"],
                        3, 1, 6, 2, 10, 3.5
                    ],
                    "line-opacity": 0.9
                }
            });

            // Show selected pipeline attributes when a line is clicked.
            map.on("click", "pipeline-lines", function (event) {
                const properties = event.features[0].properties;

                const fields = [
                    ["Company", properties.company || "Not available"],
                    ["Province", properties.province || "Not available"],
                    ["Commodity", properties.commodity || "Not available"],
                    ["Length", properties.pipeline_length_km ?? "Not available"],
                    ["Capacity", properties.capacity || "Not available"],
                    ["Status", properties.status || "Not available"],
                    ["From", properties.from_location || "Not available"],
                    ["To", properties.to_location || "Not available"]
                ];

                // Escape attribute values before inserting them into popup HTML.
                function safe(value) {
                    return String(value)
                        .replaceAll("&", "&amp;")
                        .replaceAll("<", "&lt;")
                        .replaceAll(">", "&gt;")
                        .replaceAll('"', "&quot;")
                        .replaceAll("'", "&#039;");
                }

                const pipelineName = properties.pipeline_name || "Unknown Pipeline";
                const popupHTML = `
                    <div class="popup-title">${safe(pipelineName)}</div>
                    ${fields.map(([label, value]) => `
                        <div class="popup-row">
                            <strong>${safe(label)}:</strong> ${safe(value)}
                            ${label === "Length" && value !== "Not available" ? " km" : ""}
                        </div>
                    `).join("")}
                `;

                new mapboxgl.Popup({ closeButton: true, maxWidth: "320px" })
                    .setLngLat(event.lngLat)
                    .setHTML(popupHTML)
                    .addTo(map);
            });

            // Change the cursor to indicate that pipeline lines are clickable.
            map.on("mouseenter", "pipeline-lines", function () {
                map.getCanvas().style.cursor = "pointer";
            });

            map.on("mouseleave", "pipeline-lines", function () {
                map.getCanvas().style.cursor = "";
            });

            // Fit the initial view to Canada's geographic extent.
            map.fitBounds(
                [[__MIN_LON__, __MIN_LAT__], [__MAX_LON__, __MAX_LAT__]],
                { padding: 40, duration: 1500 }
            );
        });
    </script>
</body>
</html>
"""


## 7. Build and save the HTML map

The function below converts the input data, retrieves the country boundary, computes the map extent, inserts the data into the HTML template, and writes the final file.

In [ ]:
def generate_pipeline_map(pipelines_df, output_html=OUTPUT_HTML):
    """Generate and save an interactive HTML map of Canadian pipelines."""
    print("Preparing pipeline geometries...")

    pipeline_geojson = dataframe_to_geojson(pipelines_df)
    feature_count = len(pipeline_geojson["features"])

    print(f"Successfully converted {feature_count} pipeline features.")

    # Retrieve Canada geometry and prepare its GeoJSON representation.
    canada_geometry = get_canada_boundary()

    canada_geojson = {
        "type": "Feature",
        "geometry": mapping(canada_geometry),
        "properties": {"name": "Canada"}
    }

    outside_mask = create_outside_mask(canada_geometry)

    # Calculate the geographic center and extent for the map.
    min_lon, min_lat, max_lon, max_lat = canada_geometry.bounds
    center_lon = (min_lon + max_lon) / 2
    center_lat = (min_lat + max_lat) / 2

    # Serialize Python GeoJSON objects for safe insertion into JavaScript.
    replacements = {
        "__MAPBOX_TOKEN__": html.escape(MAPBOX_TOKEN, quote=True),
        "__CENTER_LON__": str(center_lon),
        "__CENTER_LAT__": str(center_lat),
        "__CANADA_BOUNDS__": json.dumps(CANADA_BOUNDS),
        "__PIPELINE_GEOJSON__": json.dumps(pipeline_geojson),
        "__CANADA_GEOJSON__": json.dumps(canada_geojson),
        "__OUTSIDE_MASK__": json.dumps(outside_mask),
        "__MIN_LON__": str(min_lon),
        "__MIN_LAT__": str(min_lat),
        "__MAX_LON__": str(max_lon),
        "__MAX_LAT__": str(max_lat)
    }

    html_content = MAP_HTML_TEMPLATE
    for placeholder, value in replacements.items():
        html_content = html_content.replace(placeholder, value)

    # Write the standalone interactive map.
    with open(output_html, "w", encoding="utf-8") as file:
        file.write(html_content)

    absolute_path = os.path.abspath(output_html)

    print("\nMap generated successfully!")
    print(f"Output: {absolute_path}")
    print(f"Pipeline features: {feature_count}")

    return output_html


## 8. Load your pipeline data

Upload your cleaned CSV, GeoJSON, or other source file and create `pipelines_df`. If your data is already loaded in the notebook, assign it to `pipelines_df` here.

For example, if you have a CSV with WKT geometry strings:

```python
pipelines_df = pd.read_csv("cleaned_pipelines.csv")
```

If your geometries are in a GeoPackage or another GIS format, load them with GeoPandas instead.


In [ ]:
# TODO: Load or assign your cleaned pipeline DataFrame.
# Example:
# pipelines_df = pd.read_csv("cleaned_pipelines.csv")

# Confirm that the expected geometry column is present before generating the map.
assert "pipelines_df" in globals(), (
    "Load your cleaned data into a DataFrame named `pipelines_df` before continuing."
)
assert "geometry" in pipelines_df.columns, (
    "The input DataFrame must contain a 'geometry' column."
)

print(f"Input records: {len(pipelines_df):,}")
print(f"Available columns: {len(pipelines_df.columns)}")


## 9. Generate the map

Run this cell after loading `pipelines_df` and configuring your Mapbox token.

In [ ]:
map_file = generate_pipeline_map(pipelines_df)

# In Google Colab, display a clickable link to the generated HTML file.
try:
    from IPython.display import FileLink, display
    display(FileLink(map_file))
except ImportError:
    print(f"Map saved to: {map_file}")


## 10. View the map in Google Colab (optional)

The HTML file can be downloaded and opened in a browser. You can also embed it in the notebook output using an iframe.


In [ ]:
from IPython.display import IFrame, display

display(IFrame(src=map_file, width="100%", height=700))
